# Data Exploration

> Initial exploration of the provided CSV datasets (`movies.csv` and `ratings.csv`) to understand the data and schema before building the backend.


In [ ]:
import pandas as pd
import os

DATA_DIR = os.path.join(os.getcwd(), 'movie_ratings')

Data directory: /Users/anuragsubedi/Desktop/codebase/optamo-movie-ratings/movie_ratings
Files: ['output_sample.json', 'ratings.csv', 'movies.csv', 'ratings_sample.csv']


## 1. Movies Dataset (`movies.csv`)

**Schema:** `movieId` (int PK), `title` (string), `genres` (pipe-delimited string)


In [2]:
movies_df = pd.read_csv(os.path.join(DATA_DIR, 'movies.csv'))
print(f"Shape: {movies_df.shape}")
print(f"Columns: {list(movies_df.columns)}")
print(f"Dtypes:\n{movies_df.dtypes}")
movies_df.head(10)


Shape: (62423, 3)
Columns: ['movieId', 'title', 'genres']
Dtypes:
movieId    int64
title        str
genres       str
dtype: object


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [3]:
# Check for nulls and duplicates
print(f"Null counts:\n{movies_df.isnull().sum()}")
print(f"\nDuplicate movieIds: {movies_df['movieId'].duplicated().sum()}")
print(f"Unique movies: {movies_df['movieId'].nunique()}")


Null counts:
movieId    0
title      0
genres     0
dtype: int64

Duplicate movieIds: 0
Unique movies: 62423


### 1.1 Title Parsing - Extracting Name and Year

Titles follow the pattern `Movie Name (YYYY)`. We need to extract the year for the API response.


In [4]:
import re

def parse_title(title):
    """Extract clean name and release year from title like 'Toy Story (1995)'."""
    match = re.search(r'\((\d{4})\)\s*$', str(title).strip())
    if match:
        year = int(match.group(1))
        name = re.sub(r'\s*\(\d{4}\)\s*$', '', str(title).strip())
        return name, year
    return str(title).strip(), None

# Test parsing
samples = ['Toy Story (1995)', 'Grumpier Old Men (1995)', "Singin' in the Rain (1952)", 'A Girl Thing (2001)', 'Some Movie Without Year']
for s in samples:
    name, year = parse_title(s)
    print(f"  '{s}' → name='{name}', year={year}")


  'Toy Story (1995)' → name='Toy Story', year=1995
  'Grumpier Old Men (1995)' → name='Grumpier Old Men', year=1995
  'Singin' in the Rain (1952)' → name='Singin' in the Rain', year=1952
  'A Girl Thing (2001)' → name='A Girl Thing', year=2001
  'Some Movie Without Year' → name='Some Movie Without Year', year=None


In [5]:
# Apply parsing to all movies
movies_df[['name', 'release_year']] = movies_df['title'].apply(lambda t: pd.Series(parse_title(t)))

print(f"Movies without year: {movies_df['release_year'].isnull().sum()}")
print(f"Year range: {movies_df['release_year'].min()} — {movies_df['release_year'].max()}")
movies_df[['movieId', 'title', 'name', 'release_year', 'genres']].head(10)


Movies without year: 412
Year range: 1874.0 — 2019.0


,movieId,title,name,release_year,genres
0,1,Toy Story (1995),Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Jumanji,1995.0,Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Grumpier Old Men,1995.0,Comedy|Romance
3,4,Waiting to Exhale (1995),Waiting to Exhale,1995.0,Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995.0,Comedy
5,6,Heat (1995),Heat,1995.0,Action|Crime|Thriller
6,7,Sabrina (1995),Sabrina,1995.0,Comedy|Romance
7,8,Tom and Huck (1995),Tom and Huck,1995.0,Adventure|Children
8,9,Sudden Death (1995),Sudden Death,1995.0,Action
9,10,GoldenEye (1995),GoldenEye,1995.0,Action|Adventure|Thriller


### 1.2 Genre Analysis

Genres are pipe-delimited (e.g., `"Adventure|Animation|Children"`). Let's explore the unique genres.


In [6]:
# Explode genres to see distribution
all_genres = movies_df['genres'].str.split('|').explode()
genre_counts = all_genres.value_counts()
print(f"Unique genre values: {len(genre_counts)}")
print(f"\nGenre distribution:")
print(genre_counts.to_string())


Unique genre values: 20

Genre distribution:
genres
Drama                 25606
Comedy                16870
Thriller               8654
Romance                7719
Action                 7348
Horror                 5989
Documentary            5605
Crime                  5319
(no genres listed)     5062
Adventure              4145
Sci-Fi                 3595
Children               2935
Animation              2929
Mystery                2925
Fantasy                2731
War                    1874
Western                1399
Musical                1054
Film-Noir               353
IMAX                    195


In [7]:
# Movies with no genres listed
no_genre = movies_df[movies_df['genres'] == '(no genres listed)']
print(f"Movies with '(no genres listed)': {len(no_genre)}")
no_genre.head()


Movies with '(no genres listed)': 5062


,movieId,title,genres,name,release_year
15881,83773,Away with Words (San tiao ren) (1999),(no genres listed),Away with Words (San tiao ren),1999.0
16060,84768,Glitterbug (1994),(no genres listed),Glitterbug,1994.0
16351,86493,"Age of the Earth, The (A Idade da Terra) (1980)",(no genres listed),"Age of the Earth, The (A Idade da Terra)",1980.0
16491,87061,Trails (Veredas) (1978),(no genres listed),Trails (Veredas),1978.0
17404,91246,Milky Way (Tejút) (2007),(no genres listed),Milky Way (Tejút),2007.0


## 2. Ratings Dataset (`ratings.csv`)

**Schema:** `userId` (int), `movieId` (int FK), `rating` (float), `timestamp` (unix int)

> This file is ~678 MB with 25M rows. 


In [8]:
# Load ratings — this takes ~15-20 seconds
print("Loading ratings.csv... (this may take a moment)")
ratings_df = pd.read_csv(os.path.join(DATA_DIR, 'ratings.csv'))
print(f"Shape: {ratings_df.shape}")
print(f"Columns: {list(ratings_df.columns)}")
print(f"Dtypes:\n{ratings_df.dtypes}")
print(f"\nMemory usage: {ratings_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
ratings_df.head()


Loading ratings.csv... (this may take a moment)


Shape: (25000095, 4)
Columns: ['userId', 'movieId', 'rating', 'timestamp']
Dtypes:
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

Memory usage: 800.0 MB


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [9]:
# Basic stats
print(f"Null counts:\n{ratings_df.isnull().sum()}")
print(f"\nUnique users: {ratings_df['userId'].nunique():,}")
print(f"Unique movies rated: {ratings_df['movieId'].nunique():,}")
print(f"\nRating value distribution:")
print(ratings_df['rating'].value_counts().sort_index())


Null counts:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64



Unique users: 162,541


Unique movies rated: 59,047

Rating value distribution:


rating
0.5     393068
1.0     776815
1.5     399490
2.0    1640868
2.5    1262797
3.0    4896928
3.5    3177318
4.0    6639798
4.5    2200539
5.0    3612474
Name: count, dtype: int64


In [10]:
# Rating statistics
print(f"Rating range: {ratings_df['rating'].min()} — {ratings_df['rating'].max()}")
print(f"Mean rating: {ratings_df['rating'].mean():.3f}")
print(f"Median rating: {ratings_df['rating'].median():.1f}")


Rating range: 0.5 — 5.0
Mean rating: 3.534


Median rating: 3.5
